# Convert row indices \(S\) to `x[i,j,k]`, `y[i,j,k]`, and `z[i,j,k]`

This notebook supports arbitrary positive integers `m`, `n`, `p`, and `r`.

For each layer `k`, the global variable order is

```text
[vec(x[:,:,k]); vec(y[:,:,k]); vec(z[:,:,k])]
```

where

```text
x[:,:,k] has size m × n
y[:,:,k] has size n × p
z[:,:,k] has size p × m
```

Julia uses column-major vectorization. The row indices in `S` are assumed to be 1-based.

In [1]:
using DelimitedFiles

In [3]:
function row_to_xyz(
    row::Integer;
    m::Integer,
    n::Integer,
    p::Integer,
    r::Integer,
)
    @assert m > 0 "m must be positive."
    @assert n > 0 "n must be positive."
    @assert p > 0 "p must be positive."
    @assert r > 0 "r must be positive."

    x_size = m * n
    y_size = n * p
    z_size = p * m
    layer_size = x_size + y_size + z_size
    total_size = r * layer_size

    @assert 1 <= row <= total_size (
        "Row $row is outside the valid range 1:$total_size."
    )

    zero_based_row = Int(row) - 1
    k = div(zero_based_row, layer_size) + 1
    local_offset = mod(zero_based_row, layer_size) + 1

    if local_offset <= x_size
        block = :x
        local_index = local_offset
        number_of_rows = m
        number_of_columns = n
    elseif local_offset <= x_size + y_size
        block = :y
        local_index = local_offset - x_size
        number_of_rows = n
        number_of_columns = p
    else
        block = :z
        local_index = local_offset - x_size - y_size
        number_of_rows = p
        number_of_columns = m
    end

    i = mod(local_index - 1, number_of_rows) + 1
    j = div(local_index - 1, number_of_rows) + 1

    @assert 1 <= j <= number_of_columns

    variable = "$(block)[$i,$j,$k]"

    return (
        row = Int(row),
        variable = variable,
        block = block,
        i = i,
        j = j,
        k = k,
        local_offset = local_offset,
        local_index = local_index,
    )
end

row_to_xyz (generic function with 1 method)

In [5]:
function S_to_XYZ(
    S;
    m::Integer,
    n::Integer,
    p::Integer,
    r::Integer,
    output_file::AbstractString = "vars.txt",
    sort_rows::Bool = true,
    remove_duplicates::Bool = true,
)
    rows = Int.(collect(S))

    if remove_duplicates
        rows = unique(rows)
    end

    if sort_rows
        sort!(rows)
    end

    records = [
        row_to_xyz(
            row;
            m = m,
            n = n,
            p = p,
            r = r,
        )
        for row in rows
    ]

    open(output_file, "w") do io
        for record in records
            println(io, record.variable)
        end
    end

    println("Number of input rows: ", length(rows))
    println("Variables written to: ", abspath(output_file))

    return records
end

S_to_XYZ (generic function with 1 method)

## Direct input

Edit `m`, `n`, `p`, `r`, and `S`, then run the cell.

In [7]:
m = 4
n = 4
p = 4
r = 48

48

In [9]:
# Replace this example by your own 1-based row index set S.
S = [
    69,
    70,
    77,
    78,
    83,
    84,
    87,
    88,
]

8-element Vector{Int64}:
 69
 70
 77
 78
 83
 84
 87
 88

In [11]:
records = S_to_XYZ(
    S;
    m = m,
    n = n,
    p = p,
    r = r,
    output_file = "vars_test718.txt",
)

getproperty.(records, :variable)

Number of input rows: 8
Variables written to: C:\Users\21593\vars_test718.txt


8-element Vector{String}:
 "y[1,2,2]"
 "y[2,2,2]"
 "y[1,4,2]"
 "y[2,4,2]"
 "z[3,1,2]"
 "z[4,1,2]"
 "z[3,2,2]"
 "z[4,2,2]"

In [ ]:
# m = 4
# n = 4
# p = 4
# r = 48

In [13]:
S1 = vec(Int.(readdlm("positive_gap_set_80_S_1_test.txt")))

166-element Vector{Int64}:
   69
   70
   77
   78
   83
   84
   87
   88
  135
  136
  143
  144
  153
    ⋮
 2055
 2056
 2063
 2064
 2099
 2100
 2107
 2108
 2249
 2251
 2253
 2255

In [15]:
records = S_to_XYZ(
    S1;
    m = m,
    n = n,
    p = p,
    r = r,
    output_file = "vars_test166_718.txt",)

Number of input rows: 166
Variables written to: C:\Users\21593\vars_test166_718.txt


166-element Vector{@NamedTuple{row::Int64, variable::String, block::Symbol, i::Int64, j::Int64, k::Int64, local_offset::Int64, local_index::Int64}}:
 (row = 69, variable = "y[1,2,2]", block = :y, i = 1, j = 2, k = 2, local_offset = 21, local_index = 5)
 (row = 70, variable = "y[2,2,2]", block = :y, i = 2, j = 2, k = 2, local_offset = 22, local_index = 6)
 (row = 77, variable = "y[1,4,2]", block = :y, i = 1, j = 4, k = 2, local_offset = 29, local_index = 13)
 (row = 78, variable = "y[2,4,2]", block = :y, i = 2, j = 4, k = 2, local_offset = 30, local_index = 14)
 (row = 83, variable = "z[3,1,2]", block = :z, i = 3, j = 1, k = 2, local_offset = 35, local_index = 3)
 (row = 84, variable = "z[4,1,2]", block = :z, i = 4, j = 1, k = 2, local_offset = 36, local_index = 4)
 (row = 87, variable = "z[3,2,2]", block = :z, i = 3, j = 2, k = 2, local_offset = 39, local_index = 7)
 (row = 88, variable = "z[4,2,2]", block = :z, i = 4, j = 2, k = 2, local_offset = 40, local_index = 8)
 (row = 135, vari

## Optional detailed output

The returned `records` also stores the row number, block, indices, and layer.

In [ ]:
# for record in records
#     println(
#         record.row,
#         "\t",
#         record.variable,
#         "\tblock=",
#         record.block,
#         "\ti=",
#         record.i,
#         "\tj=",
#         record.j,
#         "\tk=",
#         record.k,
#     )
# end
